# Fine-Tuning DeepSeek R1 for Mental Health Chatbot

This notebook fine-tunes the DeepSeek R1 model using LoRA on the ESConv dataset for emotional support conversations, supporting the 'AI Agent Psychiatrist for Mental Health' project aligned with SDG 3.4.2.


## Which tools & packages will we be using today?

Packages we're going to be using throughout this walkthrough will be

- `unsloth`: Efficient fine-tuning and inference for LLMs — Specifically we will be using:
    - `FastLanguageModel` module to optimize inference & fine-tuning
    - `get_peft_model` to enable LoRa (Low-Rank Adaptation) fine-tuning
- `peft`: Supports LoRA-based fine-tuning for large models.
- Different Hugging Face modules:
    - `transformers` from HuggingFace to work with our fine-tuning data and handle different model tasks
    - `trl` Transformer Reinforcement Learning from HuggingFace which allows for supervised fine-tuning of the model — we will use the `SFFTrainer` wrapper
    - `datasets` to fetch reasoning datasets from the Hugging Face Hub
- `torch`: Deep learning framework used for training
- `wandb`: Provides access to weights and biases for tracking our fine-tuning experiment

## Before we get started — how to access the Hugging Face and Weights & Biases API

### Set GPU accelerator
We are using Kaggle Notebooks because we have access to free GPUs. To enable GPU access, press on Settings > Accelerator > GPU T4 x2

### How to access the Hugging Face API

1. Register to Huggin Face if you have not already
2. Go to [Hugging Face Tokens](https://huggingface.co/settings/tokens).
3. Click **"New Token"**.
4. Select **read/write** permissions if needed.
5. Copy your **API key**.

### Weights & Biases API key**
1. Sign up at [Weights & Biases](https://wandb.ai/site).
2. Go to [W&B Settings](https://wandb.ai/settings).
3. Copy your **API key** from the "API Keys" section.

### Add the API keys to Kaggle Notebooks
1. Press on Add-ons > Secrets
2. Add the API keys under `Hugging_Face_Token` and `wnb` respectively

You can now use this code to retrieve your API keys

```py
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hugging_face_token = user_secrets.get_secret("Hugging_Face_Token")
wnb_token = user_secrets.get_secret("wnb")
```

# Check Cuda

In [ ]:
import torch

In [ ]:
torch.cuda.is_available()

True

In [ ]:
torch.cuda.device_count()

1

In [ ]:
torch.cuda.get_device_name()

'NVIDIA L40S'

## Set API keys for Hugging Face and Weights & Biases

In [ ]:
import os

In [ ]:
os.environ['HUGGING_FACE_HUB_TOKEN'] = 'your_hugging_face_token'
os.environ['WANDB_API_KEY'] = 'your_wandb_token

## Install relevant packages

In [ ]:
%%capture

!pip install unsloth # install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git # Also get the latest version Unsloth!

In [ ]:
!pip install unsloth transformers datasets trl torch wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 169.0 MB/s eta 0:00:00


## Import all relevant packages throughout this walkthrough

In [ ]:
!pip show torch

Name: torch
Version: 2.7.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3-Clause
Location: /system/conda/miniconda3/envs/cloudspace/lib/python3.10/site-packages
Requires: filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-cufile-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvtx-cu12, sympy, triton, typing-extensions
Required-by: accelerate, bitsandbytes, cut-cross-entropy, lightning, litdata, peft, pytorch-lightning, sentence-transformers, torchmetrics, torchvision, unsloth_zoo, xformers


In [ ]:
!pip uninstall torch torchvision torchaudio -y

Found existing installation: torch 2.7.0
Uninstalling torch-2.7.0:
  Successfully uninstalled torch-2.7.0
Found existing installation: torchvision 0.22.0
Uninstalling torchvision-0.22.0:
  Successfully uninstalled torchvision-0.22.0


In [ ]:
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.1/799.1 MB 111.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 111.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 200.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 115.2 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 121.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 203.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 137.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 190.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 211.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 207.6 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 228.6 MB/s eta 0:00:0000:

In [ ]:
!pip uninstall unsloth -y
!pip uninstall bitsandbytes -y

Found existing installation: unsloth 2025.6.3
Uninstalling unsloth-2025.6.3:
  Successfully uninstalled unsloth-2025.6.3
Found existing installation: bitsandbytes 0.46.0
Uninstalling bitsandbytes-0.46.0:
  Successfully uninstalled bitsandbytes-0.46.0


In [ ]:
# Modules for fine-tuning
from unsloth import FastLanguageModel
import torch # Import PyTorch
from trl import SFTTrainer # Trainer for supervised fine-tuning (SFT)
from unsloth import is_bfloat16_supported # Checks if the hardware supports bfloat16 precision
# Hugging Face modules
from huggingface_hub import login # Lets you login to API
from transformers import TrainingArguments # Defines training hyperparameters
from datasets import load_dataset # Lets you load fine-tuning datasets
# Import weights and biases
import wandb

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
!pip install python-dotenv  # hanya perlu 1x

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="/teamspace/studios/this_studio/fine-tuning-deepseek-psychiatrist/.env")

True

## Create API keys and login to Hugging Face and Weights and Biases

In [ ]:
# Initialize Hugging Face & WnB tokens
# user_secrets = UserSecretsClient() # from kaggle_secrets import UserSecretsClient
# hugging_face_token = user_secrets.get_secret("Hugging_Face_Token")
# wnb_token = user_secrets.get_secret("wnb")

# # Login to Hugging Face
# login(hugging_face_token) # from huggingface_hub import login

# Login to WnB
# wandb.login(key=wnb_token) # import wandb

hugging_face_token = os.environ['HUGGING_FACE_HUB_TOKEN']
wnb_token = os.environ['WANDB_API_KEY']

# # Login to Hugging Face
login(hugging_face_token)
# Login to WnB
wandb.login(key=wnb_token)

run = wandb.init(
    project='Fine-tune-DeepSeek-R1-Distill-Llama-8B on  Mental Health Dataset',
    job_type="training",
    anonymous="allow"
)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /teamspace/studios/this_studio/.netrc
wandb: Currently logged in as: asus-yudi2022 (asus-yudi2022-haluoleo-land-group) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Loading DeepSeek R1 and the Tokenizer

**What are we doing in this step?**

In this step, we **load the DeepSeek R1 model and its tokenizer** using `FastLanguageModel.from_pretrained()`. We also **configure key parameters** for efficient inference and fine-tuning. We will be using a distilled 8B version of R1 for faster computation.  

**Key parameters explained**
```py
max_seq_length = 2048  # Define the maximum sequence length a model can handle (i.e., number of tokens per input)
dtype = None  # Default data type (usually auto-detected)
load_in_4bit = True  # Enables 4-bit quantization – a memory-saving optimization
```

**Intuition behind 4-bit quantization**

Imagine compressing a **high-resolution image** to a smaller size—**it takes up less space but still looks good enough**. Similarly, **4-bit quantization reduces the precision of model weights**, making the model **smaller and faster while keeping most of its accuracy**. Instead of storing precise **32-bit or 16-bit numbers**, we compress them into **4-bit values**. This allows **large language models to run efficiently on consumer GPUs** without needing massive amounts of memory.

In [ ]:
# Set parameters
max_seq_length = 2048 # Define the maximum sequence length a model can handle (i.e. how many tokens can be processed at once)
dtype = None # Set to default
load_in_4bit = True # Enables 4 bit quantization — a memory saving optimization

# Load the DeepSeek R1 model and tokenizer using unsloth — imported using: from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/DeepSeek-R1-Distill-Llama-8B",  # Load the pre-trained DeepSeek R1 model (8B parameter version)
    max_seq_length=max_seq_length, # Ensure the model can process up to 2048 tokens at once
    dtype=dtype, # Use the default data type (e.g., FP16 or BF16 depending on hardware support)
    load_in_4bit=load_in_4bit, # Load the model in 4-bit quantization to save memory
    token=hugging_face_token, # Use hugging face token
)

==((====))==  Unsloth 2025.6.3: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    NVIDIA L40S. Num GPUs = 1. Max memory: 44.527 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/53.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

## Testing DeepSeek R1 on a medical use-case before fine-tuning


### Defining a system prompt
To create a prompt style for the model, we will define a system prompt and include placeholders for the question and response generation. The prompt will guide the model to think step-by-step and provide a logical, accurate response.

In [ ]:
# Define a system prompt under prompt_style
prompt_style = """Below is a question from a user seeking mental health support.
Respond as a psychiatrist would, providing empathetic and helpful advice.

### Instruction:
You are a compassionate and helpful psychiatrist AI agent specializing in mental health support, especially for suicide prevention and emotional well-being.
As an AI Psychiatrist, please answer questions about mental health, psychiatric disorders, emotional support, and psychotherapy in a compassionate and professional manner.

### Question:
{}

### Response:
{}
"""

### Running inference on the model

In this step, we **test the DeepSeek R1 model** by providing a **medical question** and generating a response.  
The process involves the following steps:

1. **Define a test question** related to a medical case.
2. **Format the question using the structured prompt (`prompt_style`)** to ensure the model follows a logical reasoning process.
3. **Tokenize the input and move it to the GPU (`cuda`)** for faster inference.
4. **Generate a response using the model**, specifying key parameters like `max_new_tokens=1200` (limits response length).
5. **Decode the output tokens back into text** to obtain the final readable answer.

In [ ]:
# Creating a test medical question for inference
question = "I've been feeling really down lately and having thoughts of harming myself. What should I do?"
# test_question = "I have so many issues to address. I have a history of trauma, anxiety, and depression. Is it possible to handle all of these in counseling?"
# Enable optimized inference mode for Unsloth models (improves speed and efficiency)
FastLanguageModel.for_inference(model)  # Unsloth has 2x faster inference!

# Format the question using the structured prompt (`prompt_style`) and tokenize it
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")  # Convert input to PyTorch tensor & move to GPU

# Generate a response using the model
outputs = model.generate(
    input_ids=inputs.input_ids, # Tokenized input question
    attention_mask=inputs.attention_mask, # Attention mask to handle padding
    max_new_tokens=1200, # Limit response length to 1200 tokens (to prevent excessive output)
    use_cache=True, # Enable caching for faster inference
)

# Decode the generated output tokens into human-readable text
response = tokenizer.batch_decode(outputs)

# Extract and print only the relevant response part (after "### Response:")
print(response[0].split("### Response:")[1])



Okay, so the user is reaching out because they're feeling really down and having thoughts of harming themselves. That's a heavy burden, and I need to respond in a way that's empathetic and helpful without overwhelming them. First, I should acknowledge their feelings and let them know they're not alone. It's important to validate their emotions to build trust.

I should encourage them to talk to someone they trust, like a family member or friend, because emotional support can be really powerful. But I also need to mention professional help because that's crucial for someone who might be struggling with mental health issues.

I should mention specific resources like the National Suicide Prevention Lifeline or a crisis text line because having concrete steps can make it easier for them to take action. It's also good to suggest reaching out to a mental health professional, like a therapist or psychiatrist, because they can provide tailored support and possibly necessary treatment.

I sho

>**Before starting fine-tuning — why are we fine-tuning in the first place?**
>
> Even without fine-tuning, our model successfully generated a chain of thought and provided reasoning before delivering the final answer. The reasoning process is encapsulated within the `<think>` `</think>` tags. So, why do we still need fine-tuning? The reasoning process, while detailed, was long-winded and not concise. Additionally, we want the final answer to be consistent in a certain style.



## Fine-tuning step by step

### Step 1 — Update the system prompt
We will slightly change the prompt style for processing the dataset by adding the third placeholder for the complex chain of thought column. `</think>`

In [ ]:
# Updated training prompt style to add </think> tag
train_prompt_style = """Below is a question from a user seeking mental health support.
Respond as a psychiatrist would, providing empathetic and helpful advice.

### Instruction:
You are a compassionate and helpful psychiatrist AI agent specializing in mental health support, especially for suicide prevention and emotional well-being.
As an AI Psychiatrist, please answer questions about mental health, psychiatric disorders, emotional support, and psychotherapy in a compassionate and professional manner.

### Question:
{}

### Response:
{}
"""

### Step 2 — Download the fine-tuning dataset and format it for fine-tuning

We will use the Medical O1 Reasoninng SFT found here on [Hugging Face](https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT). From the authors: This dataset is used to fine-tune HuatuoGPT-o1, a medical LLM designed for advanced medical reasoning. This dataset is constructed using GPT-4o, which searches for solutions to verifiable medical problems and validates them through a medical verifier.

In [ ]:
# Download the dataset using Hugging Face — function imported using from datasets import load_dataset
dataset = load_dataset("nbertagnolli/counsel-chat", split = "train", trust_remote_code=True)
dataset

Repo card metadata block was not found. Setting CardData to empty.


Dataset({
    features: ['questionID', 'questionTitle', 'questionText', 'questionLink', 'topic', 'therapistInfo', 'therapistURL', 'answerText', 'upvotes', 'views'],
    num_rows: 2775
})

In [ ]:
def non_missing(example):
    return (
        example.get("questionText") is not None
        and example.get("answerText") is not None
        and str(example["questionText"]).strip() != ""
        and str(example["answerText"]).strip() != ""
    )

dataset = dataset.filter(non_missing)

In [ ]:
print(dataset)

Dataset({
    features: ['questionID', 'questionTitle', 'questionText', 'questionLink', 'topic', 'therapistInfo', 'therapistURL', 'answerText', 'upvotes', 'views'],
    num_rows: 2612
})


In [ ]:
# Show an entry from the dataset
dataset[1]

{'questionID': 0,
 'questionTitle': 'Do I have too many issues for counseling?',
 'questionText': 'I have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac.    I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years.\n   I’ve never had counseling about any of this. Do I have too many issues to address in counseling?',
 'questionLink': 'https://counselchat.com/questions/do-i-have-too-many-issues-for-counseling',
 'topic': 'depression',
 'therapistInfo': 'Jason Lynch, MS, LMHC, LCAC, ADSIndividual & Couples Therapy',
 'therapistURL': 'https://counselchat.com/therapists/jason-lynch-ms-lmhc-lcac-ads',
 'answerText': 'I\'ve never heard of someone having "too many issues" for therapy to be effective. A competent therapist will assist you in identifying the root causes of your problems and treat those first. If the underlying issues are 

>**Next step is to structure the fine-tuning dataset according to train prompt style—why?**
>
> - Each question is paired with chain-of-thought reasoning and the final response.
> - Ensures every training example follows a consistent pattern.
> - Prevents the model from continuing beyond the expected response lengt by adding the EOS token.

In [ ]:
# We need to format the dataset to fit our prompt training style
EOS_TOKEN = tokenizer.eos_token  # Define EOS_TOKEN which the model when to stop generating text during training
EOS_TOKEN

'<｜end▁of▁sentence｜>'

In [ ]:
# Define formatting prompt function
def formatting_prompts_func(examples):  # Takes a batch of dataset examples as input
    inputs = examples["questionText"]       # Extracts the mental health question from the dataset
    outputs = examples["answerText"]      # Extracts the final model-generated response (answer)

    texts = []  # Initializes an empty list to store the formatted prompts

    # Iterate over the dataset, formatting each question and response
    for input, output in zip(inputs, outputs):
        text = train_prompt_style.format(input, output) + EOS_TOKEN  # Insert values into prompt template & append EOS token
        texts.append(text)  # Add the formatted text to the list

    return {
        "text": texts,  # Return the newly formatted dataset with a "text" column containing structured prompts
    }

In [ ]:
# Update dataset formatting
dataset_finetune = dataset.map(formatting_prompts_func, batched = True)
dataset_finetune["text"][0]

Map:   0%|          | 0/2612 [00:00<?, ? examples/s]

'Below is a question from a user seeking mental health support.\nRespond as a psychiatrist would, providing empathetic and helpful advice.\n\n### Instruction:\nYou are a compassionate and helpful psychiatrist AI agent specializing in mental health support, especially for suicide prevention and emotional well-being.\nAs an AI Psychiatrist, please answer questions about mental health, psychiatric disorders, emotional support, and psychotherapy in a compassionate and professional manner.\n\n### Question:\nI have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac.    I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years.\n   I’ve never had counseling about any of this. Do I have too many issues to address in counseling?\n\n### Response:\nIt is very common for\xa0people to have multiple issues that they want to (and need to) address i

### Step 3 — Setting up the model using LoRA

**An intuitive explanation of LoRA**

Large language models (LLMs) have **millions or even billions of weights** that determine how they process and generate text. When fine-tuning a model, we usually update all these weights, which **requires massive computational resources and memory**.

LoRA (**Low-Rank Adaptation**) allows to fine-tune efficiently by:

- Instead of modifying all weights, **LoRA adds small, trainable adapters** to specific layers.  
- These adapters **capture task-specific knowledge** while leaving the original model unchanged.  
- This reduces the number of trainable parameters **by more than 90%**, making fine-tuning **faster and more memory-efficient**.  

Think of an LLM as a **complex factory**. Instead of rebuilding the entire factory to produce a new product, LoRA **adds small, specialized tools** to existing machines. This allows the factory to adapt quickly **without disrupting its core structure**.

For a more technical explanation, check out this tutorial by [Sebastian Raschka](https://www.youtube.com/watch?v=rgmJep4Sb4&t).

Below, we will use the `get_peft_model()` function which stands for Parameter-Efficient Fine-Tuning — this function wraps the base model (`model`) with LoRA modifications, ensuring that only specific parameters are trained.

In [ ]:
# Apply LoRA (Low-Rank Adaptation) fine-tuning to the model
model_lora = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank: Determines the size of the trainable adapters (higher = more parameters, lower = more efficiency)
    target_modules=[  # List of transformer layers where LoRA adapters will be applied
        "q_proj",   # Query projection in the self-attention mechanism
        "k_proj",   # Key projection in the self-attention mechanism
        "v_proj",   # Value projection in the self-attention mechanism
        "o_proj",   # Output projection from the attention layer
        "gate_proj",  # Used in feed-forward layers (MLP)
        "up_proj",    # Part of the transformer’s feed-forward network (FFN)
        "down_proj",  # Another part of the transformer’s FFN
    ],
    lora_alpha=16,  # Scaling factor for LoRA updates (higher values allow more influence from LoRA layers)
    lora_dropout=0,  # Dropout rate for LoRA layers (0 means no dropout, full retention of information)
    bias="none",  # Specifies whether LoRA layers should learn bias terms (setting to "none" saves memory)
    use_gradient_checkpointing="unsloth",  # Saves memory by recomputing activations instead of storing them (recommended for long-context fine-tuning)
    random_state=3407,  # Sets a seed for reproducibility, ensuring the same fine-tuning behavior across runs
    use_rslora=False,  # Whether to use Rank-Stabilized LoRA (disabled here, meaning fixed-rank LoRA is used)
    loftq_config=None,  # Low-bit Fine-Tuning Quantization (LoFTQ) is disabled in this configuration
)

Unsloth 2025.6.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Now, we initialize `SFTTrainer`, a supervised fine-tuning trainer from `trl` (Transformer Reinforcement Learning), to fine-tune our model efficiently on a dataset.

In [ ]:
# Initialize the fine-tuning trainer — Imported using from trl import SFTTrainer
trainer = SFTTrainer(
    model=model_lora,  # The model to be fine-tuned
    tokenizer=tokenizer,  # Tokenizer to process text inputs
    train_dataset=dataset_finetune,  # Dataset used for training
    dataset_text_field="text",  # Specifies which field in the dataset contains training text
    max_seq_length=max_seq_length,  # Defines the maximum sequence length for inputs
    dataset_num_proc=2,  # Uses 2 CPU threads to speed up data preprocessing

    # Define training arguments
    args=TrainingArguments(
        per_device_train_batch_size=2,  # Number of examples processed per device (GPU) at a time
        gradient_accumulation_steps=4,  # Accumulate gradients over 4 steps before updating weights
        num_train_epochs=3, # Full fine-tuning run
        warmup_steps=5,  # Gradually increases learning rate for the first 5 steps
        max_steps=600,  # Limits training to 60 steps (useful for debugging; increase for full fine-tuning)
        learning_rate=2e-4,  # Learning rate for weight updates (tuned for LoRA fine-tuning)
        fp16=not is_bfloat16_supported(),  # Use FP16 (if BF16 is not supported) to speed up training
        bf16=is_bfloat16_supported(),  # Use BF16 if supported (better numerical stability on newer GPUs)
        logging_steps=10,  # Logs training progress every 10 steps
        optim="adamw_8bit",  # Uses memory-efficient AdamW optimizer in 8-bit mode
        weight_decay=0.01,  # Regularization to prevent overfitting
        lr_scheduler_type="linear",  # Uses a linear learning rate schedule
        seed=3407,  # Sets a fixed seed for reproducibility
        output_dir="outputs",  # Directory where fine-tuned model checkpoints will be saved
    ),
)


Unsloth: Tokenizing ["text"]:   0%|          | 0/2612 [00:00<?, ? examples/s]

## Step 4 — Model training!

This should take around 30 to 40 minutes — we can then check out our training results on Weights and Biases

In [ ]:
# Start the fine-tuning process
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,612 | Num Epochs = 2 | Total steps = 600
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040/8,000,000,000 (0.52% trained)


Step,Training Loss
10,1.728900
20,1.747200
30,1.769300
40,1.745000
50,1.768400
60,1.733400
70,1.611200
80,1.684600
90,1.717500
100,1.744400


In [ ]:
# Save the fine-tuned model
wandb.finish()

train/epoch,▁▁▁▁▂▁▁▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
train/global_step,▁▁▁▁▂▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇███
train/grad_norm,█▂▁▁▂▃▄▃▄▃▃▂▂▂▃▃▂▂▃▃▃▃▂▆▄▄▅▇▄▅▅▄▅▄▄▅▅▄▅▆
train/learning_rate,▆▅▄▂▁████▇▇▇▇▇▆▆▆▅▅▅▅▅▅▅▄▄▄▃▃▃▃▃▃▂▂▂▁▁▁▁
train/loss,█▇▇▇▆▆▆▆▄▅▅▅▅▄▅▄▃▄▄▅▄▅▅▄▃▂▂▂▂▁▂▂▂▂▂▂▁▁▂▂
total_flos,9.468662481125376e+16
train/epoch,1.83614
train/global_step,600
train/grad_norm,0.86334
train/learning_rate,0.0
train/loss,1.3971


## Step 5 — Run model inference after fine-tuning

In [ ]:
question = "I've been feeling really down lately and having thoughts of harming myself. What should I do?"
# test_question = "I have so many issues to address. I have a history of trauma, anxiety, and depression. Is it possible to handle all of these in counseling?"

# Load the inference model using FastLanguageModel (Unsloth optimizes for speed)
FastLanguageModel.for_inference(model_lora)  # Unsloth has 2x faster inference!

# Tokenize the input question with a specific prompt format and move it to the GPU
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# Generate a response using LoRA fine-tuned model with specific parameters
outputs = model_lora.generate(
    input_ids=inputs.input_ids,          # Tokenized input IDs
    attention_mask=inputs.attention_mask, # Attention mask for padding handling
    max_new_tokens=1200,                  # Maximum length for generated response
    use_cache=True,                        # Enable cache for efficient generation
)

# Decode the generated response from tokenized format to readable text
response = tokenizer.batch_decode(outputs)

# Extract and print only the model's response part after "### Response:"
print(response[0].split("### Response:")[1])



It is always important to reach out to someone when you are feeling down.  When you are having thoughts of harming yourself, it is important to get help immediately.  I would recommend that you go to your primary care physician to rule out any medical conditions.  If you do not have a therapist, I would recommend that you find one to speak with to discuss your thoughts.  There are many options of therapists that you can contact.  There are many free and low-cost resources as well.  If you are in a crisis, you can call 1-800-273-8255 and someone will be able to talk with you.  They can also help you to find a therapist in your area.  There are also crisis centers that you can contact if you are having thoughts of harming yourself.
<｜end▁of▁sentence｜>


In [ ]:
question = "I've been feeling really down lately and having thoughts of harming myself. What should I do?"
# test_question = "I have so many issues to address. I have a history of trauma, anxiety, and depression. Is it possible to handle all of these in counseling?"

# Tokenize the input question with a specific prompt format and move it to the GPU
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# Generate a response using LoRA fine-tuned model with specific parameters
outputs = model_lora.generate(
    input_ids=inputs.input_ids,          # Tokenized input IDs
    attention_mask=inputs.attention_mask, # Attention mask for padding handling
    max_new_tokens=1200,                  # Maximum length for generated response
    use_cache=True,                        # Enable cache for efficient generation
)

# Decode the generated response from tokenized format to readable text
response = tokenizer.batch_decode(outputs)

# Extract and print only the model's response part after "### Response:"
print(response[0].split("### Response:")[1])



Although this is a very serious question, I think it is important to remember that it is possible to recover from depression.  In the short term, it is important to get in touch with your feelings, understand them, and identify the source of your depression.  It is also important to surround yourself with people who you trust and who will listen to you.  You may also want to consider seeing a counselor or therapist to help you work through the difficult emotions you are experiencing.  In the long term, it is important to identify the sources of your depression and learn to change the ways in which you handle and respond to those triggers.  You may also want to consider cognitive therapy to learn how to change your thought patterns and learn new coping skills to help you manage your emotions.  There is hope!
<｜end▁of▁sentence｜>


## Saving the model locally

In [ ]:
model_lora.save_pretrained("DeepSeek-R1-Psychiatrist1-Lora-notbit")
tokenizer.save_pretrained("DeepSeek-R1-Psychiatrist1-Lora-notbit")

('DeepSeek-R1-Psychiatrist1-Lora-notbit/tokenizer_config.json',
 'DeepSeek-R1-Psychiatrist1-Lora-notbit/special_tokens_map.json',
 'DeepSeek-R1-Psychiatrist1-Lora-notbit/tokenizer.json')

In [ ]:
print("Memulai proses merge model...")
model_lora.merge_and_unload()
print("Model berhasil di-merge.")
merged_model_path = "DeepSeek-R1-Psychiatrist-Merged-8bit"
print(f"Menyimpan model yang sudah di-merge ke folder: {merged_model_path}...")
model_lora.save_pretrained(merged_model_path)
tokenizer.save_pretrained(merged_model_path)
print("Model dan tokenizer berhasil disimpan!")

Memulai proses merge model...


/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/peft/tuners/lora/bnb.py:351: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Model berhasil di-merge.
Menyimpan model yang sudah di-merge ke folder: DeepSeek-R1-Psychiatrist-Merged-8bit...
Model dan tokenizer berhasil disimpan!


## Pushing the model to Hugging Face Hub

In [ ]:
new_model_online = "Ryuukiy/DeepSeek-R1-Psychiatrist-Lora"
model_lora.push_to_hub(new_model_online)
tokenizer.push_to_hub(new_model_online)

README.md:   0%|          | 0.00/626 [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Saved model to https://huggingface.co/Ryuukiy/DeepSeek-R1-Psychiatrist-Lora


  0%|          | 0/1 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]